# Cuaderno 0: Marco Teórico, Diseño Cuasi-Experimental, Formulación de Hipótesis y Verificación del Entorno

**Proyecto:** Impacto del Gasto Educativo Contratado (SECOP) y del Presupuesto General de la Nación (PGN) sobre la Calidad Educativa Oficial en Colombia (2019–2025).

**Autor:** Santiago Hurtado Martínez — Aprendizaje Estadístico (APES), Escuela Colombiana de Ingeniería Julio Garavito.

---

### Arquitectura del Diseño Empírico
- **Población Objetivo:** Estudiantes regulares de grado undécimo en **colegios oficiales de Calendario A** evaluados en el Examen de Estado ICFES Saber 11 ($N = 2.948.410$ evaluados únicos entre 2019 y 2025).
- **Ventana del Panel Longitudinal:** Contratación educativa territorial suscrita entre **2019 y 2023** ($t-1$) evaluada sobre el rendimiento académico oficial entre **2020 y 2024** ($t$), conformando un panel de $G = 33$ departamentos ($N = 161$ observaciones efectivas) y un panel de verificación municipal de $1.094$ municipios ($5.457$ observaciones).
- **Corte Transversal de Mediano Plazo (2025):** Acervo acumulado de contratación educativa territorial (2019–2023) y de inversión del PGN Educación (2019–2024) frente al desempeño Saber 11 Oficial en 2025 ($N = 33$ departamentos).
- **Estrategia de Identificación:** Modelos de Efectos Fijos en Dos Vías (*Two-Way Fixed Effects* — TWFE), Mínimos Cuadrados Ponderados (WLS) por matrícula oficial del año base pre-pandemia ($N_{i,2019}^{\text{oficial}}$), Estimador en Primeras Diferencias ($\Delta Y_{it}$), Prueba de Exogeneidad Dinámica de Wooldridge ($X_{i,t+1}$) e inferencia robusta mediante *Wild Cluster Bootstrap* (1.999 réplicas de Rademacher).

## Hipótesis Estructurales del Estudio

### Hipótesis 1 (H1): Sesgo de Selección en el Confinamiento (2020) y Rezago Pedagógico en el Retorno Presencial (2021)
> *Una vez corregida la duplicación de registros estudiantiles del período 2019-4, la contracción de evaluados durante el confinamiento de 2020 fue moderada ($-5,22\%$ nacional; $-4,10\%$ en colegios oficiales de Calendario A), induciendo un aumento aparente del puntaje promedio por composición socioeconómica ($+1,34$ puntos), mientras que el deterioro real del aprendizaje se materializó en el retorno general de 2021 ($-1,83$ puntos respecto a 2019 y $-3,17$ puntos respecto a 2020), seguido de una recuperación sostenida hacia 2024.*

### Hipótesis 2 (H2): Acotamiento de la Elasticidad Interanual del Gasto Educativo Contratado
> *Una vez depurados los procesos multi-beneficiario de SECOP I y los contratos repetidos de SECOP II, excluidas las entidades del orden nacional domiciliadas en Bogotá D.C. y normalizado el gasto por la matrícula oficial fija del año base ($N_{i,2019}^{\text{oficial}}$), la elasticidad interanual ($t-1$) del gasto educativo contratado per cápita sobre el puntaje Saber 11 oficial está estrechamente acotada alrededor de cero ($|\beta| < 0,8$ puntos), dado que la nómina docente oficial se financia mediante el Sistema General de Participaciones (SGP) por fuera del sistema contractual SECOP.*

### Hipótesis 3 (H3): Heterogeneidad Funcional del Gasto (CAPEX vs. Bienestar) y Mecanismo de Retención Escolar
> *Al desagregar el gasto educativo contratado en dos componentes mutuamente excluyentes —Inversión Física/Dotación (CAPEX) y Bienestar/Operación Escolar (PAE y transporte escolar)—, el rubro de Bienestar actúa primordialmente como un mecanismo de protección social que sostiene el piso de la distribución cognitiva (percentil 25, `punt_global_p25`) y reduce la tasa de deserción intra-anual oficial.*

### Hipótesis 4 (H4): Primacía de la Gobernanza Contractual y la Ejecución Efectiva sobre el Volumen Licitado
> *La eficiencia en el desembolso y pago efectivo de los contratos educativos (`tasa_ejecucion_media` en SECOP II) se asocia positivamente con el desempeño académico ($+22,91$ puntos, $p = 0,0185$), mientras que la dependencia excesiva de la contratación directa exhibe una asociación negativa y estadísticamente significativa dentro de los departamentos ($\beta = -17,19$, $p = 0,0052$).*

### Hipótesis 5 (H5): Dominancia de Factores Estructurales y Regularización Lineal en la Predicción de Variaciones Anuales ($\Delta Y_{it}$)
> *Al predecir la variación interanual del logro educativo ($\Delta Y_{it} = Y_{it} - Y_{i,t-1}$) utilizando exclusivamente información rezagada bajo validación temporal (`Rolling Origin`) y espacial (`GroupKFold`), los modelos lineales penalizados (Ridge y ElasticNet) superan a los ensambles de árboles de decisión, y la descomposición SHAP confirma que la dinámica anual está gobernada por la tasa de deserción, la pobreza multidimensional y la reversión a la media.*

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, HTML

def resolve_project_root() -> Path:
    """Resuelve de forma portable la raíz del proyecto o del paquete Entregable."""
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent, cwd.parent.parent]:
        if (candidate / 'Outputs' / 'dataset_maestro_departamental_v4.parquet').exists():
            return candidate
    return cwd.parent

BASE_DIR = resolve_project_root()
DATA_DIR = BASE_DIR / 'Data'
OUTPUTS_DIR = BASE_DIR / 'Outputs'
TABLAS_DIR = OUTPUTS_DIR / 'Tablas'
FIGURAS_DIR = OUTPUTS_DIR / 'Figuras'

matriz_hipotesis = pd.DataFrame([
    {
        'Código': 'H1',
        'Dimensión Analítica': 'Dinámica Pandémica y Composición Demográfica',
        'Hipótesis Sustantiva': 'Selección moderada en 2020 (-4,10% oficial) y bache pedagógico real en 2021 (-1,83 pts vs 2019)',
        'Variables y Métricas': 'n_estudiantes, punt_global_mean, punt_global_p25 (2019–2025)',
        'Método de Contraste': 'Descomposición censal longitudinal por cohorte y departamento (Cuaderno 02)'
    },
    {
        'Código': 'H2',
        'Dimensión Analítica': 'Efecto Causal Primario del Gasto Contratado',
        'Hipótesis Sustantiva': 'Elasticidad interanual acotada en torno a cero y exogeneidad estricta dinámica',
        'Variables y Métricas': 'log_gasto_edu_pc_lag1, log_gasto_edu_pc_lead1, log_pgn_educacion_pc_lag1',
        'Método de Contraste': 'TWFE, WLS (peso N_2019), Primeras Diferencias, Test de Wooldridge y Wild Cluster Bootstrap (Cuaderno 03)'
    },
    {
        'Código': 'H3',
        'Dimensión Analítica': 'Mecanismos de Transmisión y Retención',
        'Hipótesis Sustantiva': 'El gasto en Bienestar (PAE/transporte) protege el percentil 25 y reduce la deserción oficial',
        'Variables y Métricas': 'log_gasto_capex_pc_lag1, log_gasto_bienestar_pc_lag1, punt_global_p25, tasa_desercion',
        'Método de Contraste': 'Modelos TWFE WLS por componente sobre P25 y tasa de deserción intra-anual (Cuaderno 03)'
    },
    {
        'Código': 'H4',
        'Dimensión Analítica': 'Gobernanza Contractual y Ejecución',
        'Hipótesis Sustantiva': 'Efecto negativo de la contratación directa y positivo de la tasa de pago efectivo',
        'Variables y Métricas': 'pct_contratacion_directa_lag1, hhi_proveedores_edu_lag1, tasa_ejecucion_media',
        'Método de Contraste': 'Panel TWFE WLS (2020–2024) y Regresión Transversal HC3 en SECOP II (Cuaderno 03)'
    },
    {
        'Código': 'H5',
        'Dimensión Analítica': 'Aprendizaje Estadístico Fuera de Muestra',
        'Hipótesis Sustantiva': 'Superioridad de regularización lineal sobre árboles en ΔY y nulo ΔRMSE incremental de SECOP',
        'Variables y Métricas': 'delta_punt_global, RMSE ponderado, Skill Score, valores SHAP (TreeExplainer)',
        'Método de Contraste': 'Validación Rolling Origin y GroupKFold con IC 95% Bootstrap por clusters (Cuaderno 03)'
    }
])
print('=== Matriz Estructural de Hipótesis y Estrategia de Identificación ===')
display(HTML(matriz_hipotesis.to_html(index=False)))

# Verificación automática de disponibilidad de activos de datos en el entorno actual
activos_requeridos = [
    ('Catálogo Territorial DIVIPOLA', DATA_DIR / 'divipola_oficial.csv'),
    ('Deflactores IPC Base 2019', OUTPUTS_DIR / 'ipc_deflactores.csv'),
    ('Microdatos SECOP Educativo Territorial (353.273 contratos)', OUTPUTS_DIR / 'secop_depurado_v2.parquet'),
    ('Agregado SECOP Departamental', OUTPUTS_DIR / 'secop_departamental.parquet'),
    ('Serie Censal ICFES Saber 11 Oficial Cal. A', OUTPUTS_DIR / 'icfes_depurado_v2.parquet'),
    ('Indicadores Oficiales TerriData Educación (2018-2024)', OUTPUTS_DIR / 'terri_educacion_departamental.parquet'),
    ('PGN Educación Regionalizado Deflactado (2019-2025)', OUTPUTS_DIR / 'pgn_departamental_consolidado_v2.csv'),
    ('Índice de Pobreza Multidimensional DANE (2019-2024)', OUTPUTS_DIR / 'ipm_departamental_2019_2024.csv'),
    ('Contratos PAE Deduplicados (2019-2022)', OUTPUTS_DIR / 'pae_departamental_2019_2022.parquet'),
    ('Panel Maestro Departamental (231 filas × 98 cols)', OUTPUTS_DIR / 'dataset_maestro_departamental_v4.parquet'),
    ('Panel Maestro Municipal (7.714 filas × 45 cols)', OUTPUTS_DIR / 'dataset_maestro_municipal_v4.parquet')
]
inventario = []
for nombre, ruta in activos_requeridos:
    existe = ruta.exists()
    tam_kb = round(ruta.stat().st_size / 1024, 1) if existe else 0.0
    inventario.append({'Activo de Datos': nombre, 'Ruta Relativa': f'{ruta.parent.name}/{ruta.name}', 'Tamaño (KB)': tam_kb, 'Estado': '✔ Disponible' if existe else '✘ Faltante'})
df_inv = pd.DataFrame(inventario)
assert (df_inv['Estado'] == '✔ Disponible').all(), 'Faltan archivos requeridos en el entorno'
print(f'\n=== Verificación de Integridad del Paquete de Datos (Raíz activa: {BASE_DIR.name}) ===')
display(df_inv)

=== Matriz Estructural de Hipótesis y Estrategia de Identificación ===


Código,Dimensión Analítica,Hipótesis Sustantiva,Variables y Métricas,Método de Contraste
H1,Dinámica Pandémica y Composición Demográfica,"Selección moderada en 2020 (-4,10% oficial) y bache pedagógico real en 2021 (-1,83 pts vs 2019)","n_estudiantes, punt_global_mean, punt_global_p25 (2019–2025)",Descomposición censal longitudinal por cohorte y departamento (Cuaderno 02)
H2,Efecto Causal Primario del Gasto Contratado,Elasticidad interanual acotada en torno a cero y exogeneidad estricta dinámica,"log_gasto_edu_pc_lag1, log_gasto_edu_pc_lead1, log_pgn_educacion_pc_lag1","TWFE, WLS (peso N_2019), Primeras Diferencias, Test de Wooldridge y Wild Cluster Bootstrap (Cuaderno 03)"
H3,Mecanismos de Transmisión y Retención,El gasto en Bienestar (PAE/transporte) protege el percentil 25 y reduce la deserción oficial,"log_gasto_capex_pc_lag1, log_gasto_bienestar_pc_lag1, punt_global_p25, tasa_desercion",Modelos TWFE WLS por componente sobre P25 y tasa de deserción intra-anual (Cuaderno 03)
H4,Gobernanza Contractual y Ejecución,Efecto negativo de la contratación directa y positivo de la tasa de pago efectivo,"pct_contratacion_directa_lag1, hhi_proveedores_edu_lag1, tasa_ejecucion_media",Panel TWFE WLS (2020–2024) y Regresión Transversal HC3 en SECOP II (Cuaderno 03)
H5,Aprendizaje Estadístico Fuera de Muestra,Superioridad de regularización lineal sobre árboles en ΔY y nulo ΔRMSE incremental de SECOP,"delta_punt_global, RMSE ponderado, Skill Score, valores SHAP (TreeExplainer)",Validación Rolling Origin y GroupKFold con IC 95% Bootstrap por clusters (Cuaderno 03)



=== Verificación de Integridad del Paquete de Datos (Raíz activa: Entregable) ===


,Activo de Datos,Ruta Relativa,Tamaño (KB),Estado
0,Catálogo Territorial DIVIPOLA,Data/divipola_oficial.csv,41.1,✔ Disponible
1,Deflactores IPC Base 2019,Outputs/ipc_deflactores.csv,0.1,✔ Disponible
2,Microdatos SECOP Educativo Territorial (353.27...,Outputs/secop_depurado_v2.parquet,62396.4,✔ Disponible
3,Agregado SECOP Departamental,Outputs/secop_departamental.parquet,39.5,✔ Disponible
4,Serie Censal ICFES Saber 11 Oficial Cal. A,Outputs/icfes_depurado_v2.parquet,44.2,✔ Disponible
5,Indicadores Oficiales TerriData Educación (201...,Outputs/terri_educacion_departamental.parquet,15.3,✔ Disponible
6,PGN Educación Regionalizado Deflactado (2019-2...,Outputs/pgn_departamental_consolidado_v2.csv,41.3,✔ Disponible
7,Índice de Pobreza Multidimensional DANE (2019-...,Outputs/ipm_departamental_2019_2024.csv,4.3,✔ Disponible
8,Contratos PAE Deduplicados (2019-2022),Outputs/pae_departamental_2019_2022.parquet,4.7,✔ Disponible
9,Panel Maestro Departamental (231 filas × 98 cols),Outputs/dataset_maestro_departamental_v4.parquet,190.3,✔ Disponible
